# KKBOX Churn Prediction

## Feature Engineering Pipeline

### Objective
The goal of this notebook is to transform multiple relational datasets into a single customer-level feature table for predicting customer churn.

This notebook builds a memory-efficient feature engineering pipeline for the KKBOX Churn Prediction dataset.

The pipeline performs:

- Loading raw datasets
- Memory optimization
- Data cleaning
- Feature engineering
- Dataset merging
- Final dataset export

The output of this notebook is used for model development with CatBoost, LightGBM, XGBoost, and Logistic Regression.

# Dataset

This project is based on the **KKBOX Churn Prediction Challenge** dataset from Kaggle.

The notebook was developed using **Version 2** of the competition dataset.

Due to the dataset size (approximately 9 GB), the raw data is **not included** in this repository.

To reproduce the project, obtain the dataset from the official Kaggle competition and place the files inside the `data/` directory.

Required files:

- train_v2.csv
- transactions_v2.csv
- members_v3.csv
- user_logs_v2.csv

## 1. Loading raw datasets

The notebook automatically detects whether it is running:

- Locally (using the `data/` directory)
- In a Kaggle Notebook (using the mounted competition dataset)

This allows the same notebook to run in both environments without modification.


In [ ]:
# ======================================================
# Project Configuration
# ======================================================

RANDOM_STATE = 42

EXPORT_DIR = Path("processed")

EXPORT_DIR.mkdir(exist_ok=True)

In [ ]:
# ======================
# imports
# ======================
from pathlib import Path
#------------------------
# using polars because it's better and faster in handling massive dataset's then pandas
import polars as pl

In [ ]:
# ==========================================================
# Detect Execution Environment
# ==========================================================

KAGGLE_DATA_DIR = Path("/kaggle/input/kkbox-churn-prediction-challenge")
LOCAL_DATA_DIR = Path("data")

if KAGGLE_DATA_DIR.exists():
    DATA_DIR = KAGGLE_DATA_DIR
    print("Running in Kaggle environment.")
elif LOCAL_DATA_DIR.exists():
    DATA_DIR = LOCAL_DATA_DIR
    print("Running in local environment.")
else:
    raise FileNotFoundError(
        "Dataset directory not found. "
        "Place the data in ./data or run the notebook in Kaggle."
    )

In [ ]:
# ==========================================================
# Load Raw Datasets
# ==========================================================

train = pl.read_csv(DATA_DIR / "train_v2.csv")
members = pl.read_csv(DATA_DIR / "members_v3.csv")
transactions = pl.read_csv(DATA_DIR / "transactions_v2.csv")
user_logs = pl.read_csv(DATA_DIR / "user_logs_v2.csv")

print("Datasets loaded successfully.")

## 3. Memory Optimization

To reduce RAM usage, numeric columns are automatically downcast to the smallest safe datatype.

This significantly decreases memory consumption while preserving data integrity.

In [ ]:
# ==========================================================
# Memory Optimization Utilities
# ==========================================================

def get_optimal_numeric_type(col_name: str, min_val: float, max_val: float, is_float: bool) -> pl.DataType:
    """Determines the absolute smallest, safest data type for a numeric range."""
    if is_float:
        # Floats default to 64-bit; 32-bit is perfectly precise for standard metrics
        return pl.Float32

    # Check for unsigned integer boundaries (0 or positive numbers only)
    if min_val >= 0:
        if max_val <= 255:                  return pl.UInt8
        elif max_val <= 65_535:             return pl.UInt16
        elif max_val <= 4_294_967_295:       return pl.UInt32
        else:                               return pl.UInt64
    else:
        # Check for signed integer boundaries (contains negative numbers)
        if min_val >= -128 and max_val <= 127:                      return pl.Int8
        elif min_val >= -32_768 and max_val <= 32_767:              return pl.Int16
        elif min_val >= -2_147_483_648 and max_val <= 2_147_483_647: return pl.Int32
        else:                                                       return pl.Int64

def auto_shrink_dataframe(df: pl.DataFrame, df_name: str) -> pl.DataFrame:
    """Dynamically analyzes boundaries and shrinks all numeric columns safely."""
    orig_size = df.estimated_size() / (1024 ** 2)
    transforms = []

    # Loop over columns and analyze schemas
    for col_name, dtype in df.schema.items():
        if dtype.is_numeric():
            # Extract actual min and max values from the data
            stats = df.select([pl.col(col_name).min().alias("min"), pl.col(col_name).max().alias("max")]).row(0)
            min_v, max_v = stats[0], stats[1]

            # Skip checking if column is entirely null
            if min_v is None or max_v is None:
                continue

            is_float = dtype.is_float()
            optimal_type = get_optimal_numeric_type(col_name, min_v, max_v, is_float)

            # Only queue transformation if it actually saves space
            if optimal_type != dtype:
                transforms.append(pl.col(col_name).cast(optimal_type))

    # Apply transformations if any were found
    if transforms:
        df_shrunk = df.with_columns(transforms)
    else:
        df_shrunk = df

    new_size = df_shrunk.estimated_size() / (1024 ** 2)
    print(f"🔹 [{df_name}] Shrunk from {orig_size:.2f} MB to {new_size:.2f} MB ({((orig_size - new_size)/orig_size)*100:.1f}% reduction)")
    return df_shrunk

def optimize_all_data():
    print("⚡ Starting Intelligent Dynamic Downcasting...\n")

    # Process files individually to keep peak RAM usage low
    train_clean = auto_shrink_dataframe(pl.read_csv('train_v2.csv'), "train_v2.csv")
    transcation_clean = auto_shrink_dataframe(pl.read_csv('transactions_v2.csv'), "transactions_v2.csv")
    logs_clean = auto_shrink_dataframe(pl.read_csv('user_logs_v2.csv'), "user_logs_v2.csv")
    members_clean = auto_shrink_dataframe(pl.read_csv('members_v3.csv'), "members_v3.csv")

    # Calculate total new RAM usage

    total_new_ram = (train_clean.estimated_size() + transcations_clean.estimated_size() +
                     logs_clean.estimated_size() + members_clean.estimated_size()) / (1024 ** 2)
    print("\n==================================================")
    print(f"📈 Total Optimized Data Size in RAM: {total_new_ram:.2f} MB")
    print("==================================================")

    return train_clean, transcations_clean, logs_clean, members_clean

# Execute optimization safely
train_clean, transactions_clean, user_logs_clean, members_clean = optimize_all_data()

## Feature engineering

### Transaction Features


In [ ]:
# ============================================================
# KKBOX TRANSACTION FEATURE ENGINEERING
# Input: transactions_clean
# ============================================================

# -----------------------------
# 1) Parse Dates & Sort Chronologically
# -----------------------------
transactions_clean = transactions_clean.with_columns([
    pl.col('transaction_date').cast(pl.String).str.to_date('%Y%m%d'),
    pl.col('membership_expire_date').cast(pl.String).str.to_date('%Y%m%d')
]).sort(["msno", "transaction_date", "membership_expire_date"])

# ------------------------------------------------------------
# 2) Save Global Cutoff / Reference Date
# ------------------------------------------------------------
reference_date = transactions_clean.select(
    pl.col("transaction_date").max()
).item()

# ============================================================
# 3) Transaction-Level Feature Engineering
# ============================================================
transactions_clean = (
    transactions_clean
    .with_columns([
        # Absolute discount amount
        (pl.col("plan_list_price") - pl.col("actual_amount_paid"))
        .cast(pl.Float32)
        .alias("discount"),

        # Ratio of paid amount to list price (captures promo depth)
        pl.when(pl.col("plan_list_price") > 0)
        .then(pl.col("actual_amount_paid") / pl.col("plan_list_price"))
        .otherwise(0)
        .fill_nan(0)
        .fill_null(0)
        .cast(pl.Float32)
        .alias("paid_ratio"),

        # Binary flag if a discount was applied
        pl.when(pl.col("actual_amount_paid") < pl.col("plan_list_price"))
        .then(1)
        .otherwise(0)
        .cast(pl.Int8)
        .alias("used_discount"),

        # Price paid per plan day
        pl.when(pl.col("payment_plan_days") > 0)
        .then(pl.col("actual_amount_paid") / pl.col("payment_plan_days"))
        .otherwise(0)
        .fill_nan(0)
        .fill_null(0)
        .cast(pl.Float32)
        .alias("price_per_day"),
    ])
    .with_columns([
        # Days since previous transaction
        pl.col("transaction_date")
        .diff()
        .dt.total_days()
        .fill_null(0)
        .over("msno")
        .cast(pl.Int32)
        .alias("days_since_last_transaction"),

        # Renewal gap (Current transaction date - previous expiry date)
        (
            pl.col("transaction_date")
            - pl.col("membership_expire_date").shift(1).over("msno")
        )
        .dt.total_days()
        .fill_null(0)
        .cast(pl.Int32)
        .alias("renewal_gap_days"),

        # Membership duration purchased in this transaction
        (pl.col("membership_expire_date") - pl.col("transaction_date"))
        .dt.total_days()
        .cast(pl.Int32)
        .alias("membership_duration"),

        # Transaction-to-transaction alterations (for variance tracking)
        pl.col("actual_amount_paid").diff().over("msno").fill_null(0).cast(pl.Float32).alias("payment_change"),
        pl.col("payment_plan_days").diff().over("msno").fill_null(0).cast(pl.Int32).alias("plan_change"),
    ])
)

# ============================================================
# 4) Customer-Level Aggregation
# ============================================================
 transaction_features = (
     transactions_clean
    .group_by("msno", maintain_order=True)  # maintain_order guarantees chronological first/last accuracy
    .agg([
        # A) TRANSACTION COUNT
        pl.len().cast(pl.Int32).alias("transaction_count"),

        # B) ACTUAL AMOUNT PAID STATS
        pl.col("actual_amount_paid").last().cast(pl.Float32).alias("last_paid"),
        pl.col("actual_amount_paid").mean().cast(pl.Float32).alias("avg_paid"),
        pl.col("actual_amount_paid").sum().cast(pl.Float32).alias("total_paid"),
        pl.col("actual_amount_paid").std().cast(pl.Float32).alias("std_paid"),

        # C) LIST PRICE STATS
        pl.col("plan_list_price").last().cast(pl.Float32).alias("last_list_price"),
        pl.col("plan_list_price").mean().cast(pl.Float32).alias("avg_list_price"),
        pl.col("plan_list_price").std().cast(pl.Float32).alias("std_list_price"),
        pl.col("plan_list_price").n_unique().cast(pl.Int16).alias("num_price_changes"),

        # D) PLAN DURATION STATS
        pl.col("payment_plan_days").last().cast(pl.Int32).alias("last_plan_days"),
        pl.col("payment_plan_days").mean().cast(pl.Float32).alias("avg_plan_days"),
        pl.col("payment_plan_days").std().cast(pl.Float32).alias("std_plan_days"),
        pl.col("payment_plan_days").n_unique().cast(pl.Int16).alias("num_plan_changes"),

        # E) DISCOUNT / PRICING STATS
        pl.col("discount").mean().cast(pl.Float32).alias("avg_discount"),
        pl.col("discount").sum().cast(pl.Float32).alias("total_discount"),
        pl.col("used_discount").sum().cast(pl.Int32).alias("num_discount_transaction"),
        pl.col("price_per_day").mean().cast(pl.Float32).alias("avg_price_per_day"),
        pl.col("paid_ratio").mean().cast(pl.Float32).alias("avg_paid_ratio"),

        # F) AUTO-RENEW / CANCELLATION
        pl.col("is_auto_renew").mean().cast(pl.Float32).alias("auto_renew_rate"),
        pl.col("is_auto_renew").sum().cast(pl.Int32).alias("num_auto_renew"),
        pl.col("is_auto_renew").last().cast(pl.Int8).alias("last_auto_renew"),
        pl.col("is_cancel").mean().cast(pl.Float32).alias("cancel_rate"),
        pl.col("is_cancel").sum().cast(pl.Int32).alias("num_cancel"),
        pl.col("is_cancel").last().cast(pl.Int8).alias("last_cancel"),

        # G) PAYMENT METHOD
        pl.col("payment_method_id").n_unique().cast(pl.Int16).alias("num_payment_method"),
        pl.col("payment_method_id").last().cast(pl.Int16).alias("last_payment_method"),

        # H) RECENCY / EXPIRY
        pl.col("membership_expire_date").last().alias("last_expire_date"),
        (pl.lit(reference_date) - pl.col("transaction_date").max()).dt.total_days().cast(pl.Int32).alias("recent_activity_days"),

        # I) RENEWAL GAP FEATURES
        pl.col("renewal_gap_days").mean().cast(pl.Float32).alias("avg_renewal_gap"),
        pl.col("renewal_gap_days").std().cast(pl.Float32).alias("std_renewal_gap"),
        pl.col("renewal_gap_days").max().cast(pl.Int32).alias("max_renewal_gap"),

        # J) MEMBERSHIP DURATION
        pl.col("membership_duration").mean().cast(pl.Float32).alias("avg_membership_duration"),

        # K) VOLATILITY STATS (Standard deviations of window differences)
        pl.col("payment_change").std().cast(pl.Float32).alias("std_payment_change"),
        pl.col("plan_change").std().cast(pl.Float32).alias("std_plan_change"),

        # L) CUSTOMER-LEVEL TEMPORAL FEATURES
        # Tenure = Last transaction date minus First transaction date
        (pl.col("transaction_date").max() - pl.col("transaction_date").min()).dt.total_days().cast(pl.Int32).alias("customer_tenure"),

        # Trends (Net shifts over customer lifetime)
        (pl.col("actual_amount_paid").last() - pl.col("actual_amount_paid").first()).cast(pl.Float32).alias("payment_trend"),
        (pl.col("payment_plan_days").last() - pl.col("payment_plan_days").first()).cast(pl.Int32).alias("plan_trend"),
        (pl.col("discount").last() - pl.col("discount").first()).cast(pl.Float32).alias("discount_trend"),

        # Renewal frequency per active subscription day span
        pl.when((pl.col("transaction_date").max() - pl.col("transaction_date").min()).dt.total_days() > 0)
        .then(pl.len() / (pl.col("transaction_date").max() - pl.col("transaction_date").min()).dt.total_days())
        .otherwise(0)
        .cast(pl.Float32)
        .alias("renewal_frequency"),
    ])
    .with_columns([
        # ----------------------------------------------------
        # 5) Safe Null Handling for Valid Variances
        # ----------------------------------------------------
        pl.col("std_paid").fill_null(0),
        pl.col("std_list_price").fill_null(0),
        pl.col("std_plan_days").fill_null(0),
        pl.col("avg_renewal_gap").fill_null(0),
        pl.col("std_renewal_gap").fill_null(0),
        pl.col("max_renewal_gap").fill_null(0),
        pl.col("avg_membership_duration").fill_null(0),
        pl.col("std_payment_change").fill_null(0),
        pl.col("std_plan_change").fill_null(0),
        pl.col("customer_tenure").fill_null(0),
        pl.col("payment_trend").fill_null(0),
        pl.col("plan_trend").fill_null(0),
        pl.col("discount_trend").fill_null(0),
        pl.col("renewal_frequency").fill_null(0),
        pl.col("avg_paid_ratio").fill_null(0),

    ]) # Closing the first .with_columns

    .with_columns([
        # ----------------------------------------------------
        # 6) Derived Target-Specific Features
        # ----------------------------------------------------
        # Higher consistency value indicates predictable subscriber patterns
        (1 / (1 + pl.col("std_renewal_gap"))).cast(pl.Float32).alias("renewal_consistency"),

        # Calculated timeline distance to final subscription expiration from reference point
        (pl.col("last_expire_date") - pl.lit(reference_date)).dt.total_days().cast(pl.Int32).alias("days_to_expiry_at_cutoff"),

        # Explicit binary check signaling past-due subscription status at timeline cutoff
        (pl.col("last_expire_date") <= pl.lit(reference_date)).cast(pl.Int8).alias("is_expired_at_cutoff"),
    ])
)

In [ ]:
# ==========================================================
# Validate Transaction Features
# ==========================================================

print(f"Transaction features shape: {transaction_features.shape}")

assert (
    transaction_features.height
    == transaction_features["msno"].n_unique()
), "Duplicate customers found in transaction_features."

null_summary = (
    transaction_features
    .null_count()
    .transpose(include_header=True, header_name="feature")
    .rename({"column_0": "null_count"})
    .filter(pl.col("null_count") > 0)
)

print(null_summary)



### Member Features


In [ ]:
# ============================================================
# KKBOX  members FEATURE ENGINEERING
# Input: members_clean
# ============================================================

reference_date = pl.lit("2017-03-31").str.to_date("%Y-%m-%d") # latest transcation date in the dataset

members_clean = members_clean.with_columns([
    pl.col('registration_init_time').cast(pl.String).str.to_date('%Y%m%d'),
    pl.when((pl.col('bd') >= 10) & (pl.col('bd') <= 90))
    .then(pl.col('bd'))
    .otherwise(None)
    .alias('bd')
]).with_columns([
    pl.col('bd').is_null().cast(pl.Int8).alias('age_missing')
]).with_columns([
    pl.col('registration_init_time').dt.year().cast(pl.Int16).alias('registration_year'),
    pl.col('registration_init_time').dt.month().cast(pl.Int8).alias('registration_month')
]).with_columns([
    (reference_date - pl.col('registration_init_time')).dt.total_days().cast(pl.Int16).alias('member_tenure_days')
]).with_columns([
    pl.col("gender")
      .fill_null("Unknown")
      .alias("gender")
])

In [ ]:
# ==========================================================
# Validate Member Features
# ==========================================================

print(f"Member features shape: {member_features.shape}")

assert (
    member_features.height
    == member_features["msno"].n_unique()
), "Duplicate customers found in member_features."

null_summary = (
    member_features
    .null_count()
    .transpose(include_header=True, header_name="feature")
    .rename({"column_0": "null_count"})
    .filter(pl.col("null_count") > 0)
)

print(null_summary)


### User log Features

In [ ]:
# Convert the 'date' column in user_logs_clean to a Polars Date type
# This ensures proper date arithmetic later in the cell.
user_logs_clean = user_logs_clean.with_columns(
    pl.col("date").cast(pl.String).str.to_date("%Y%m%d")
)

REFERENCE_DATE = pl.date(2017, 3, 31) # it's max trancation date

# ------------------------------------------------------------------------------
# Create row-wise features
# ------------------------------------------------------------------------------
user_logs_clean = user_logs_clean.with_columns([

    # Total plays on each day
    (
        pl.col("num_25") +
        pl.col("num_50") +
        pl.col("num_75") +
        pl.col("num_985") +
        pl.col("num_100")
    ).alias("total_plays"),

    # Completion rate
    (
        pl.col("num_100") /
        (
            pl.col("num_25") +
            pl.col("num_50") +
            pl.col("num_75") +
            pl.col("num_985") +
            pl.col("num_100")
        ).clip(lower_bound=1)
    ).alias("completion_rate"),

    # Skip rate
    (
        pl.col("num_25") /
        (
            pl.col("num_25") +
            pl.col("num_50") +
            pl.col("num_75") +
            pl.col("num_985") +
            pl.col("num_100")
        ).clip(lower_bound=1)
    ).alias("skip_rate"),

    # Unique song ratio
    (
        pl.col("num_unq") /
        (
            pl.col("num_25") +
            pl.col("num_50") +
            pl.col("num_75") +
            pl.col("num_985") +
            pl.col("num_100")
        ).clip(lower_bound=1)
    ).alias("unique_ratio")
])

# ------------------------------------------------------------------------------
# Aggregate user-level features
# ------------------------------------------------------------------------------
user_log_features = (
    user_logs_clean
    .group_by("msno")
    .agg([

        # Active days
        pl.len().alias("active_days"),

        # Last activity (now derived from a Date type column)
        pl.col("date").max().alias("last_log_date"),

        # Listening time
        pl.col("total_secs").sum().alias("total_secs_sum"),
        pl.col("total_secs").mean().alias("total_secs_mean"),
        pl.col("total_secs").median().alias("total_secs_median"),
        pl.col("total_secs").max().alias("total_secs_max"),
        pl.col("total_secs").min().alias("total_secs_min"),
        pl.col("total_secs").std().alias("total_secs_std"),

        # Total plays
        pl.col("total_plays").sum().alias("total_plays_sum"),
        pl.col("total_plays").mean().alias("total_plays_mean"),
        pl.col("total_plays").max().alias("total_plays_max"),

        # Playback counts
        pl.col("num_25").sum().alias("num25_sum"),
        pl.col("num_50").sum().alias("num50_sum"),
        pl.col("num_75").sum().alias("num75_sum"),
        pl.col("num_985").sum().alias("num985_sum"),
        pl.col("num_100").sum().alias("num100_sum"),

        # Unique songs
        pl.col("num_unq").sum().alias("unique_songs_sum"),
        pl.col("num_unq").mean().alias("unique_songs_mean"),

        # Average engagement metrics
        pl.col("completion_rate").mean().alias("completion_rate"),
        pl.col("skip_rate").mean().alias("skip_rate"),
        pl.col("unique_ratio").mean().alias("unique_ratio")
    ])
)

# ------------------------------------------------------------------------------
# Derived features
# ------------------------------------------------------------------------------
user_log_features = user_log_features.with_columns([

    # Average listening seconds per active day
    (pl.col("total_secs_sum") /
        pl.col("active_days")
    ).alias("avg_secs_per_day"),

    # Average plays per active day
    (pl.col("total_plays_sum") /
        pl.col("active_days")
    ).alias("avg_plays_per_day"),

    # Days since last activity
    (
        REFERENCE_DATE -
        pl.col("last_log_date") # This is now a Date type, compatible with REFERENCE_DATE
    ).dt.total_days().alias("days_since_last_log")
])

In [ ]:
# ==========================================================
# Validate User Log Features
# ==========================================================

print(f"User log features shape: {user_log_features.shape}")

assert (
    user_log_features.height
    == user_log_features["msno"].n_unique()
), "Duplicate customers found in user_log_features."

null_summary = (
    user_log_features
    .null_count()
    .transpose(include_header=True, header_name="feature")
    .rename({"column_0": "null_count"})
    .filter(pl.col("null_count") > 0)
)

print(null_summary)

## Dataset merging

In [ ]:
# =================================
# Merging all three datasets
# =================================

# Merging by train_clean

final_train = train_clean
final_train = final_train.join(
    members_clean,
    on="msno",
    how="left"
)
final_train = final_train.join(
    transaction_features,
    on="msno",
    how="left"
)
final_train = final_train.join(
    user_log_features,
    on="msno",
    how="left"
)

In [ ]:
# After merging you would have a lot of null values
# because some rows contain in three dataset's won't be in train_clean
final_train.null_count()

In [ ]:
# ==========================================================
# Handle Missing Values After Feature Merging
# ==========================================================

member_cols = [
    "registration_year",
    "registration_month",
    "member_tenure_days",
]

final_train = final_train.with_columns(

    # Member features
    *[
        pl.col(col)
        .fill_null(-1)
        .cast(pl.Int32)
        .alias(col)
        for col in member_cols
    ],

    # Categorical features
    *[
        pl.col(col)
        .fill_null("Unknown")
        .alias(col)
        for col in categorical_cols
    ],

    # Binary features
    *[
        pl.col(col)
        .fill_null(0)
        .cast(pl.Int8)
        .alias(col)
        for col in binary_cols
    ],

    # Transaction features
    *[
        pl.col(col)
        .fill_null(0)
        .alias(col)
        for col in transaction_cols
    ],

    # Listening features
    *[
        pl.col(col)
        .fill_null(0)
        .alias(col)
        for col in log_cols
    ],

    # Age
    pl.when(
        (pl.col("bd") < 10) | (pl.col("bd") > 90)
    )
    .then(None)
    .otherwise(pl.col("bd"))
    .fill_null(-1)
    .cast(pl.Int16)
    .alias("bd"),
)

# ==========================================================
# Remove Raw Date Columns
# ==========================================================

final_train = final_train.drop([
    "registration_init_time",
    "last_expire_date",
    "last_log_date",
])

In [ ]:
# ==========================================================
# Validate Final Training Dataset
# ==========================================================

print(f"Final training shape: {final_train.shape}")

assert (
    final_train.height
    == final_train["msno"].n_unique()
), "Duplicate customers found in final_train."

null_summary = (
    final_train
    .null_count()
    .transpose(include_header=True, header_name="feature")
    .rename({"column_0": "null_count"})
    .filter(pl.col("null_count") > 0)
)

print(null_summary)

## Final dataset export

In [ ]:
# ==========================================
# Memory optimization for final_train
# ==========================================


def optimize_final_train(df: pl.DataFrame) -> pl.DataFrame:
    print("⚡ Starting Intelligent Smart-Shrink for final_train...")
    orig_size = df.estimated_size() / (1024 ** 2)
    transforms = []

    for col_name, dtype in df.schema.items():
        # 1. Skip the user ID string
        if col_name == "msno":
            continue

        # 2. Handle categorical columns (like gender or string classes)
        if dtype == pl.String:
            transforms.append(pl.col(col_name).cast(pl.Categorical))
            continue

        # 3. Handle decimal features / rates / standard deviations
        if dtype.is_float():
            # Float32 is more than precise enough for rates, ratios, and averages
            transforms.append(pl.col(col_name).cast(pl.Float32))
            continue

        # 4. Handle integers based on actual mathematical boundaries
        if dtype.is_numeric():
            stats = df.select([pl.col(col_name).min().alias("min"), pl.col(col_name).max().alias("max")]).row(0)
            min_v, max_v = stats[0], stats[1]

            # Skip if the column is completely filled with Nulls
            if min_v is None or max_v is None:
                continue

            # Unsigned integers (0 and positive values only)
            if min_v >= 0:
                if max_v <= 255:                  optimal_type = pl.UInt8
                elif max_v <= 65_535:             optimal_type = pl.UInt16
                elif max_v <= 4_294_967_295:       optimal_type = pl.UInt32
                else:                               optimal_type = pl.UInt64
            # Signed integers (contains negative numbers/trends)
            else:
                if min_v >= -128 and max_v <= 127:                      optimal_type = pl.Int8
                elif min_v >= -32_768 and max_v <= 32_767:              optimal_type = pl.Int16
                elif min_v >= -2_147_483_648 and max_val <= 2_147_483_647: optimal_type = pl.Int32
                else:                                                   optimal_type = pl.Int64

            if optimal_type != dtype:
                transforms.append(pl.col(col_name).cast(optimal_type))

    # Apply all optimized datatypes safely at once
    df_shrunk = df.with_columns(transforms)
    new_size = df_shrunk.estimated_size() / (1024 ** 2)

    print("==================================================")
    print(f"📈 Original final_train RAM: {orig_size:.2f} MB")
    print(f"📉 Optimized final_train RAM: {new_size:.2f} MB")
    print(f"🎉 RAM Saved: {orig_size - new_size:.2f} MB ({((orig_size - new_size)/orig_size)*100:.1f}% reduction)")
    print("==================================================")

    return df_shrunk

# Run the smart shrink on your dataset
final_train_clean = optimize_final_train(final_train)


### Export Processed Dataset

The fully processed training dataset is exported in both CSV and
Parquet formats.

• CSV provides broad compatibility with analytics tools and spreadsheets.
• Parquet offers efficient storage, faster loading, and preserves data types,
  making it suitable for machine learning workflows and large-scale processing.


In [ ]:
# ==========================================================
# Export Final Training Dataset
# ==========================================================


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "final_train_clean.csv"
parquet_path = OUTPUT_DIR / "final_train_clean.parquet"

# Export dataset
final_train_clean.write_csv(csv_path)
final_train_clean.write_parquet(parquet_path)

print("Dataset exported successfully!")
print(f"CSV file: {csv_path}")
print(f"Parquet file: {parquet_path}")
print(f"Final dataset shape: {final_train_clean.shape}")

## Summary

This notebook transformed the raw KKBOX datasets into a single customer-level modeling dataset through extensive feature engineering.

Key outputs:

• Engineered behavioral features
• Engineered payment features
• Engineered customer history features
• Memory-optimized dataset
• Final dataset exported for model training